In [1]:
import time
import re
import pytz
import pandas as pd
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import selenium.common.exceptions as se

amazon_base = "https://www.amazon.com.tr"
amazon_login_page = "https://www.amazon.com.tr/ap/signin?openid.pape.max_auth_age=0&openid.return_to=https%3A%2F%2Fwww.amazon.com.tr%2F%3Fref_%3Dnav_ya_signin&openid.identity=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0%2Fidentifier_select&openid.assoc_handle=trflex&openid.mode=checkid_setup&openid.claimed_id=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0%2Fidentifier_select&openid.ns=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0"

TR_MONTHS = {
    "Ocak": 1, "Şubat": 2, "Mart": 3, "Nisan": 4, "Mayıs": 5, "Haziran": 6,
    "Temmuz": 7, "Ağustos": 8, "Eylül": 9, "Ekim": 10, "Kasım": 11, "Aralık": 12
}

def make_driver(detach=True, headless=False):
    chrome_options = webdriver.ChromeOptions()
    if detach:
        chrome_options.add_experimental_option("detach", True)
    if headless:
        chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1400,900")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(60)
    return driver
import threading
import os
import math


In [2]:
def build_review_url(asin: str, page: int) -> str:
    ref = "cm_cr_arp_d_viewopt_sr" if page <= 1 else f"cm_cr_arp_d_paging_btm_next_{page}"
    return (
        f"{amazon_base}/product-reviews/{asin}/ref={ref}"
        f"?ie=UTF8&reviewerType=all_reviews&sortBy=recent&pageNumber={page}"
    )

def build_product_url(asin: str) -> str:
    return f"{amazon_base}/dp/{asin}"

In [3]:
def wait_reviews_stable(driver, timeout=20, stable_rounds=3, poll=0.25):
    end = time.time() + timeout
    last = -1
    stable = 0
    while time.time() < end:
        cards = driver.find_elements(By.CSS_SELECTOR, '#cm_cr-review_list [data-hook="review"]')
        n = len(cards)
        if n == last and n > 0:
            stable += 1
            if stable >= stable_rounds:
                return n
        else:
            stable = 0
            last = n
        time.sleep(poll)
    return len(driver.find_elements(By.CSS_SELECTOR, '#cm_cr-review_list [data-hook="review"]'))


In [4]:
def safe_text(el) -> str:
    try:
        return el.text.strip()
    except Exception:
        return ""

def parse_amazon_tr_rating(text: str):
    if not text:
        return None
    m = re.search(r"üzerinden\s+(\d+(?:[.,]\d+)?)", text)
    if not m:
        m = re.search(r"(\d+(?:[.,]\d+)?)", text)
    if not m:
        return None
    return float(m.group(1).replace(",", "."))


def parse_amazon_tr_date_to_iso(date_text: str):
    if not date_text:
        return None

    m = re.search(r"(\d{1,2})\s+([A-Za-zÇĞİÖŞÜçğıöşü]+)\s+(\d{4})", date_text)
    if not m:
        return None

    day = int(m.group(1))
    mon_name = m.group(2).strip()
    year = int(m.group(3))
    mon = TR_MONTHS.get(mon_name)
    if not mon:
        return None
    return datetime(year, mon, day).date().isoformat()

In [5]:
class GracefulStopState:
    def __init__(self):
        self.event = threading.Event()
        self.last_requested_at = None

    def request_stop(self):
        self.last_requested_at = datetime.now(pytz.timezone("Europe/Istanbul")).strftime("%Y-%m-%d %H:%M:%S")
        self.event.set()

STOP_STATE = GracefulStopState()

def start_stop_listener(stop_state=STOP_STATE):
    """
    Jupyter/terminal için güvenli durdurma:
    q + ENTER yazıldığında scraping durur.
    """
    if getattr(start_stop_listener, "_started", False):
        return

    def _listen():
        print("Durdurmak için herhangi bir anda q yazıp ENTER'a bas.")
        while not stop_state.event.is_set():
            try:
                cmd = input().strip().lower()
            except EOFError:
                break
            except Exception:
                break
            if cmd == "q":
                stop_state.request_stop()
                print("\n[STOP] Durdurma isteği alındı. Mevcut ürün bitirilip veriler kaydedilecek.")
                break

    t = threading.Thread(target=_listen, daemon=True)
    t.start()
    start_stop_listener._started = True

def save_progress_csv(all_dfs, output_csv_path: str):
    if all_dfs:
        df_all = pd.concat(all_dfs, ignore_index=True)
    else:
        df_all = pd.DataFrame()

    if not df_all.empty and "review_id" in df_all.columns:
        df_all = df_all.drop_duplicates(subset=["review_id"])

    df_all.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {len(df_all)} satır kaydedildi -> {output_csv_path}")
    return df_all


In [6]:
def get_product_meta(driver, asin: str, wait_timeout: int = 15) -> dict:
    wait = WebDriverWait(driver, wait_timeout)

    product_url = build_product_url(asin)
    driver.get(product_url)

    product_title = None
    try:
        title_el = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#productTitle")))
        product_title = title_el.text.strip()
    except Exception:
        product_title = None

    product_description = None
    desc_candidates = [
        "#feature-bullets ul li span.a-list-item",
        "ul.a-unordered-list.a-vertical li span.a-list-item",
        "#productDescription",
    ]

    bullets = []
    for sel in desc_candidates:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            texts = [e.text.strip() for e in els if e.text and e.text.strip()]
            if texts:
                if sel == "#productDescription" and not bullets:
                    product_description = " ".join(texts).strip()
                    break
                else:
                    bullets.extend(texts)
        except Exception:
            continue

    if bullets:
        product_description = " | ".join(dict.fromkeys(bullets))

    return {
        "product_title": product_title,
        "product_description": product_description,
        "product_url": product_url
    }

In [7]:
def find_local_and_global_cards(driver):
    container = driver.find_element(By.CSS_SELECTOR, "#cm_cr-review_list")

    cards = container.find_elements(By.CSS_SELECTOR, 'li[data-hook="review"], div[data-hook="review"]')

    local_headers = container.find_elements(By.CSS_SELECTOR, 'h3[data-hook="arp-local-reviews-header"]')
    local_text = local_headers[0].text.strip() if local_headers else ""

    is_local_section = ("Ülke:" in local_text) or ("Türkiye" in local_text)

    if is_local_section:
        return cards, []
    else:
        return [], cards

In [8]:
def extract_rating_from_card_text(card) -> float | None:
    try:
        txt = card.get_attribute("innerText") or card.get_attribute("textContent") or ""
        txt = txt.replace("\u00a0", " ").strip()  # nbsp temizle
    except Exception:
        txt = (card.text or "").strip()

    if not txt:
        return None

    m = re.search(
        r"(\d+(?:[.,]\d+)?)\s*yıldız\s*üzerinden\s*(\d+(?:[.,]\d+)?)",
        txt,
        flags=re.IGNORECASE
    )
    if m:
        return float(m.group(2).replace(",", "."))

    m = re.search(
        r"(\d+(?:[.,]\d+)?)\s*üzerinden\s*(\d+(?:[.,]\d+)?)",
        txt,
        flags=re.IGNORECASE
    )
    if m:
        return float(m.group(2).replace(",", "."))

    m = re.search(
        r"(\d+(?:[.,]\d+)?)\s*yıldız",
        txt,
        flags=re.IGNORECASE
    )
    if m:
        return float(m.group(1).replace(",", "."))

    return None

In [9]:
def parse_review_card_to_row(card, asin: str, current_reviews_url: str, product_meta: dict = None):
    # Title
    title = None
    try:
        title_el = card.find_element(By.CSS_SELECTOR, '[data-hook="review-title"]')
        title = safe_text(title_el)
    except Exception:
        pass

    # Rating -> float
    rating_val = extract_rating_from_card_text(card)

    # Body
    body = None
    try:
        body_el = card.find_element(By.CSS_SELECTOR, '[data-hook="review-body"]')
        body = safe_text(body_el).replace("\n", " ").strip()
    except Exception:
        pass

    # Date -> ISO
    review_date_iso = None
    try:
        date_el = card.find_element(By.CSS_SELECTOR, 'span[data-hook="review-date"]')
        review_date_iso = parse_amazon_tr_date_to_iso(safe_text(date_el))
    except Exception:
        pass

    review_id = None
    try:
        review_id = card.get_attribute("id")  # R32D2E5N2T6S02
    except Exception:
        pass

    turkey_tz = pytz.timezone("Europe/Istanbul")
    collected_at = datetime.now(turkey_tz).strftime("%Y-%m-%d %H:%M")

    return {
            "review_id": review_id,
            "asin": asin,
            "title": title,
            "rating": rating_val,
            "body": body,
            "review_date": review_date_iso,
    
            # product meta
            "product_title": (product_meta or {}).get("product_title"),
            "product_description": (product_meta or {}).get("product_description"),
    
            "product_url": build_product_url(asin),
            "source": "amazon_tr",
            "review_summary": None,
            "collected_at": collected_at,
    }

In [10]:
def scrape_most_recent_local_reviews_df(
    driver,
    asin: str,
    target_n: int = 80,
    max_pages: int = 80,
    wait_timeout: int = 15,
    stop_state=STOP_STATE
):
    wait = WebDriverWait(driver, wait_timeout)
    seen = set()
    rows = []

    # --- product meta (visit product page once) ---
    try:
        product_meta = get_product_meta(driver, asin, wait_timeout=wait_timeout)
    except Exception:
        product_meta = {"product_title": None, "product_description": None}

    driver.get(build_review_url(asin, 1))

    last_url = None

    for page in range(1, max_pages + 1):
        if stop_state.event.is_set():
            print(f"[STOP] {asin} ürünü içinde {page}. sayfa öncesi durdurma algılandı.")
            break

        # --- wait for reviews to load ---
        try:
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#cm_cr-review_list")))
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '#cm_cr-review_list [data-hook="review"]')))
            wait_reviews_stable(driver, timeout=25, stable_rounds=4)
        except se.TimeoutException:
            break

        # --- prevent infinite loop if Amazon keeps us on same page ---
        if driver.current_url == last_url:
            break
        last_url = driver.current_url

        # --- get review cards (local vs global) ---
        local_cards, global_cards = find_local_and_global_cards(driver)

        # stop if global started or locals ended
        if len(global_cards) > 0 or len(local_cards) == 0:
            break

        # --- parse cards ---
        for i, card in enumerate(local_cards):
            if stop_state.event.is_set():
                print(f"[STOP] {asin} ürünü {page}. sayfada işlenirken durdurma algılandı.")
                break

            try:
                rid = card.get_attribute("id")
            except Exception:
                rid = None

            if rid and rid in seen:
                continue

            try:
                row = parse_review_card_to_row(card, asin, driver.current_url, product_meta=product_meta)
            except se.StaleElementReferenceException:
                cards2 = driver.find_elements(By.CSS_SELECTOR, '#cm_cr-review_list li[data-hook="review"], #cm_cr-review_list div[data-hook="review"]')
                if i >= len(cards2):
                    continue
                row = parse_review_card_to_row(cards2[i], asin, driver.current_url, product_meta=product_meta)
                rid = row.get("review_id")

            if rid:
                seen.add(rid)
            rows.append(row)

            if len(seen) >= target_n:
                df = pd.DataFrame(rows)
                df = df.drop_duplicates(subset=["review_id"])
                return df.head(target_n)

        if stop_state.event.is_set():
            break

        time.sleep(0.6)

        # --- click NEXT (Sonraki) instead of building URL ---
        try:
            disabled_next = driver.find_elements(By.CSS_SELECTOR, "li.a-disabled.a-last")
            if disabled_next:
                break

            next_a = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.a-last a")))
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_a)

            try:
                next_a.click()
            except Exception:
                driver.execute_script("arguments[0].click();", next_a)

        except se.TimeoutException:
            break

    df = pd.DataFrame(rows[:target_n])

    if "review_id" in df.columns:
        df = df.drop_duplicates(subset=["review_id"])
    else:
        df = df.drop_duplicates()

    return df


In [11]:
def scrape_many_asins(
    driver,
    asins,
    target_n=80,
    max_pages=80,
    output_csv_path="amazon_tr_reviews_local_only.csv",
    products_per_hour=100,
    enable_stop_listener=True,
    stop_state=STOP_STATE
):
    all_dfs = []
    seconds_per_product = 3600 / products_per_hour if products_per_hour and products_per_hour > 0 else 0
    stopped_on_asin = None

    if enable_stop_listener:
        start_stop_listener(stop_state=stop_state)

    for idx, asin in enumerate(asins, 1):
        if stop_state.event.is_set():
            stopped_on_asin = asin
            print(f"[STOP] Yeni ürüne başlamadan durdu. Sıradaki ürün: {asin}")
            break

        started_at = time.time()

        try:
            print(f"[{idx}/{len(asins)}] Scraping ASIN: {asin}")
            df = scrape_most_recent_local_reviews_df(
                driver,
                asin=asin,
                target_n=target_n,
                max_pages=max_pages,
                stop_state=stop_state
            )
            print(f"  -> got {len(df)} reviews")

            if not df.empty:
                all_dfs.append(df)

            # her ürün sonrası ara kayıt
            save_progress_csv(all_dfs, output_csv_path)

            if stop_state.event.is_set():
                stopped_on_asin = asin
                print(f"[STOP] Durdurulan ürün: {asin}")
                break

        except Exception as e:
            print(f"  !! FAILED ASIN {asin}: {type(e).__name__}: {e}")

        elapsed = time.time() - started_at
        wait_needed = max(0, seconds_per_product - elapsed)
        if wait_needed > 0 and idx < len(asins) and not stop_state.event.is_set():
            print(f"  -> rate limit için {wait_needed:.1f} sn bekleniyor")
            slept = 0.0
            while slept < wait_needed:
                if stop_state.event.is_set():
                    stopped_on_asin = asin
                    print(f"[STOP] Bekleme sırasında durdurma alındı. Son tamamlanan ürün: {asin}")
                    break
                step = min(1.0, wait_needed - slept)
                time.sleep(step)
                slept += step
            if stop_state.event.is_set():
                break

    final_df = save_progress_csv(all_dfs, output_csv_path)

    if stopped_on_asin is not None:
        print(f"[INFO] Scraping durdu. İlgili ürün: {stopped_on_asin}")

    if not final_df.empty:
        return final_df

    return pd.DataFrame(columns=[
        "asin",
        "title","rating","body","review_date",
        "product_title","product_description",
        "product_url","source","review_summary","collected_at"
    ])


In [12]:
driver = make_driver(detach=True, headless=False)

# Login (manuel)
driver.get(amazon_login_page)

In [12]:
asins = [
# Elektrikli Süpürge
"B08S787DSS", "B08P7WBBXP", "B0B6GFY5LB", "B08D6CR4N9", "B0DCC2YQP7",
"B0DSKXG7WB", "B0DCGD7Y8K", "B0DCGD7Y8K", "B0B829K872", "B07YNZWR56",

# Filtre Kahve Makinesi
"B00R04CAH0", "B01534GMCM", "B0BX9MGW6G", "B00R04CAWA", "B09WYWPY14",
"B095SRNR3H", "B01B0YQV66", "B084ZPHMS1", "B00MB6J36I", "B0741Z3WJC",

# Diş Macunu
"B07YQFT2ZV", "B004PCEQZE", "B07BB54JZ4", "B08R9CWLY1", "B07G11C7YW",
"B0BTT2SKDT", "B09SB9P5QZ", "B0DKFZ5X2V", "B0D4F6523X", "B073T33K69",

# Diş Fırçası
"B0DXPNVX74", "B089F3W17W", "B0B515Z52M", "B0B5192LBN", "B0DZ2CF8M1",
"B08GHGMD4N", "B0B51CWCN7", "B0BJMS84K6", "B086DV91S3", "B018907QLI",

# Maskara
"B0BWRTJLZ1", "B0CJ5GW8RY", "B0092QK8N2", "B0CJ5FPMN3", "B00PFCSXD4",
"B07L7Q962B", "B07FDHR2QM", "B0CJ5G8LKV", "B00C7D24IE", "B008OG7V2M",

]

In [14]:
import json

with open("03122026Amazon.txt", "r", encoding="utf-8") as f:
    links = json.load(f)

print(type(links))  # list
asins = [u.split("/dp/")[1][:10] for u in links]

<class 'list'>


In [18]:
print(len(asins))

269


In [16]:
asins.index("B07W6J2LWD")

83

In [17]:
asins = asins[83:]

In [19]:
OUTPUT_CSV_PATH = "Starting from B07W6J2LWD.csv"

df_all = scrape_many_asins(
    driver,
    asins,
    target_n=80,
    max_pages=80,
    output_csv_path=OUTPUT_CSV_PATH,
    products_per_hour=50,
    enable_stop_listener=True
)

if "review_id" in df_all.columns:
    df_all = df_all.drop("review_id", axis=1)

df_all.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
df_all.head()


Durdurmak için herhangi bir anda q yazıp ENTER'a bas.[1/269] Scraping ASIN: B07W6J2LWD

  -> got 10 reviews
[SAVE] 10 satır kaydedildi -> Starting from B07W6J2LWD.csv
  -> rate limit için 47.1 sn bekleniyor
[2/269] Scraping ASIN: B07W6JHVG8
  -> got 10 reviews
[SAVE] 20 satır kaydedildi -> Starting from B07W6J2LWD.csv
  -> rate limit için 50.3 sn bekleniyor


 q



[STOP] Durdurma isteği alındı. Mevcut ürün bitirilip veriler kaydedilecek.
[STOP] Bekleme sırasında durdurma alındı. Son tamamlanan ürün: B07W6JHVG8
[SAVE] 20 satır kaydedildi -> Starting from B07W6J2LWD.csv
[INFO] Scraping durdu. İlgili ürün: B07W6JHVG8


,asin,title,rating,body,review_date,product_title,product_description,product_url,source,review_summary,collected_at
0,B07W6J2LWD,f/p,5.0,ürün fiyatına göre gayet iyi. ses çıkartmayaca...,2026-03-03,"Logitech M171 Kablosuz Mouse, PC, Mac, Dizüstü...",Tak ve çalıştır bağlantısı: Bilgisayarınızın U...,https://www.amazon.com.tr/dp/B07W6J2LWD,amazon_tr,None,2026-03-19 11:13
1,B07W6J2LWD,Garanti Sorunu,2.0,Mouse tekerleği bozuldu değişim için logitech ...,2026-03-03,"Logitech M171 Kablosuz Mouse, PC, Mac, Dizüstü...",Tak ve çalıştır bağlantısı: Bilgisayarınızın U...,https://www.amazon.com.tr/dp/B07W6J2LWD,amazon_tr,None,2026-03-19 11:13
2,B07W6J2LWD,Çok memnun kaldık,5.0,Çok kullanışlı çok güzel çok kaliteli bir ürün...,2026-02-28,"Logitech M171 Kablosuz Mouse, PC, Mac, Dizüstü...",Tak ve çalıştır bağlantısı: Bilgisayarınızın U...,https://www.amazon.com.tr/dp/B07W6J2LWD,amazon_tr,None,2026-03-19 11:13
3,B07W6J2LWD,iyi,4.0,"Kutuda teslimat iyi, küçük isteyenler için iyi...",2026-02-28,"Logitech M171 Kablosuz Mouse, PC, Mac, Dizüstü...",Tak ve çalıştır bağlantısı: Bilgisayarınızın U...,https://www.amazon.com.tr/dp/B07W6J2LWD,amazon_tr,None,2026-03-19 11:13
4,B07W6J2LWD,Orjinal mi,1.0,Bu ürün orjinal mi emin olamadım çünkü seri no...,2026-02-25,"Logitech M171 Kablosuz Mouse, PC, Mac, Dizüstü...",Tak ve çalıştır bağlantısı: Bilgisayarınızın U...,https://www.amazon.com.tr/dp/B07W6J2LWD,amazon_tr,None,2026-03-19 11:13


In [ ]:
# [113/352] Scraping ASIN: B086CSN1D9
#   -> got 0 reviews
# [114/352] Scraping ASIN: B086DQP567
#   -> got 0 reviews
# [115/352] Scraping ASIN: B086QY2DN2
#   -> got 0 reviews
# [116/352] Scraping ASIN: B086ZSZ3MK

In [ ]:
df_all.to_csv("03122026Amazon.csv")

#### 03/13/2026 Amazon
**1325**

#### 03/14/2026 Amazon
**3358**

#### 03/12/2026 Amazon ( array.txt )
**353**

#### Bilgisayar

#### Bebek 

#### Bahce

#### Ayakkabi